In [19]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from collections import defaultdict
import random

In [2]:
habr = pd.read_csv("dataset\habr.csv")  

<>:1: SyntaxWarning: invalid escape sequence '\h'
<>:1: SyntaxWarning: invalid escape sequence '\h'
C:\Users\Альберт\AppData\Local\Temp\ipykernel_9696\1708112871.py:1: SyntaxWarning: invalid escape sequence '\h'
  habr = pd.read_csv("dataset\habr.csv")


In [5]:
habr.shape

(151904, 9)

In [6]:
habr['hubs'].nunique()

76935

In [9]:
import ast

def parse_hubs(hubs_str):
    try:
        return ast.literal_eval(hubs_str) 
    except:
        return []

habr['hubs_list'] = habr['hubs'].apply(parse_hubs)
all_hubs = [hub for sublist in habr['hubs_list'] for hub in sublist]
unique_hubs = set(all_hubs)
count = len(unique_hubs)
print(f"Уникальных: {count}")

Уникальных: 1417


In [31]:
hub_to_indices = defaultdict(list)
for idx, hubs in enumerate(habr['hubs_list']):
    for h in hubs:
        hub_to_indices[h].append(idx)

hubs_with_min10 = {hub: indices for hub, indices in hub_to_indices.items() if len(indices) >= 10}
print(f"Хабов с ≥5 статьями: {len(hubs_with_min10)}")

selected_indices = set()
for hub, indices in hubs_with_min10.items():
    chosen = random.sample(indices, 10) 
    selected_indices.update(chosen)

print(f"Выбрано уникальных статей: {len(selected_indices)}")
selected_ids = habr.loc[list(selected_indices), 'id'].tolist()

Хабов с ≥5 статьями: 957
Выбрано уникальных статей: 9332


In [32]:
pd.DataFrame({'id': selected_ids}).to_csv("selected_ids_for_embeddings.csv", index=False)

In [ ]:
selected_ids = pd.read_csv("selected_ids_for_embeddings.csv")['id'].tolist()
df = pd.read_csv("dataset\\data_with_main_info.csv")
df_selected = df[df['id'].isin(selected_ids)].copy()
print(f"Найдено {len(df_selected)} статей из {len(selected_ids)} запрошенных ID")

model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def get_sentence_embedding(text, max_len=512):
    tokens = tokenizer.encode(text, add_special_tokens=True)
    if len(tokens) <= max_len:
        inputs = tokenizer(text, return_tensors="pt", 
                           max_length=max_len, truncation=True, 
                           padding='max_length')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        attention_mask = inputs['attention_mask']
        last_hidden = outputs.last_hidden_state
        masked_emb = last_hidden * attention_mask.unsqueeze(-1)
        mean_emb = masked_emb.sum(dim=1) / attention_mask.sum(dim=1, keepdim=True)
        return mean_emb.squeeze().cpu().numpy()
    
    stride = max_len // 2
    chunks = [tokens[i:i+max_len] for i in range(0, len(tokens), stride)]
    chunk_embs = []
    for chunk in chunks:
        if len(chunk) > max_len:
            chunk = chunk[:max_len]
        attention_mask = [1] * len(chunk) + [0] * (max_len - len(chunk))
        chunk = chunk + [tokenizer.pad_token_id] * (max_len - len(chunk))
        inputs = {
            'input_ids': torch.tensor([chunk]).to(device),
            'attention_mask': torch.tensor([attention_mask]).to(device)
        }
        with torch.no_grad():
            outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state
        mask = inputs['attention_mask'].unsqueeze(-1).float()
        mean_emb = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1)
        chunk_embs.append(mean_emb.squeeze().cpu().numpy())
    article_emb = np.mean(chunk_embs, axis=0)
    return article_emb

embeddings = []
ids = []
for _, row in tqdm(df_selected.iterrows(), total=len(df_selected)):
    text = str(row['cleaned_text']) if isinstance(row['cleaned_text'], str) else ""
    if text.strip():
        emb = get_sentence_embedding(text)
    else:
        emb = np.zeros(model.config.hidden_size)
    embeddings.append(emb)
    ids.append(row['id'])

embedding_matrix = np.stack(embeddings, axis=0)
np.save("balanced_embeddings.npy", embedding_matrix)
pd.DataFrame({'id': ids}).to_csv("balanced_ids.csv", index=False)

print(f"Размер матрицы: {embedding_matrix.shape}")

Найдено 7609 статей из 9332 запрошенных ID


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
 60%|██████    | 4574/7609 [7:18:39<149:42:08, 177.57s/it]  

В итоге - прогрузили меньшее количество - потому что на домашнем компьютере крутить все это оказалось проблематично, на полпути захлебнулось просто выполнение.
Перешли к более качественным подходам с более сильными моделями - в итоге отходим от подхода с эмбедингами - будем использовать E5, FASTTEXT и генеративные модели для саммаризации - которые показали хорошие результаты.